# Lab for Lesson 2 - Model Optimization and Export for Embedded Deployment

## Importing Required Libraries

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras import layers
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import random
import sys
import ctypes
import threading
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from pathlib import Path
import seaborn as sns
import librosa
import scipy.signal as signalpy
from scipy.io import wavfile
from ansys.scadeone.core import ScadeOne
from ansys.scadeone.core.svc.pywrapper.python_wrapper import PythonWrapper

## Utilities

In [ ]:
print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)
print(tf.config.list_physical_devices())

## Creating the data pipeline for training and testing starting from the already generated spectrograms

Train **nine models**: the same three CNN configurations for each of the three splits. It does not evaluate the test sets.

Before running, prepare three independent copies of the PNG dataset. For each split, place all images in `training/Normal` and `training/Murmur`. Create the same two class subfolders under `validation` and `testing`, and leave them empty. Keep the original dataset separate.

The split cell **moves** images from your training copies. Run it only once per copy. To train again without moving more files, set `MOVE_IMAGES = False` before running all cells. Training again replaces model and text files with the same names.


### Set the paths and training options

In [ ]:
# Replace these paths with your own.
train_70 = Path(r"<path to 70_10_20\training>")
val_70   = Path(r"<path to 70_10_20\validation>")
test_70  = Path(r"<path to 70_10_20\testing>")

train_85 = Path(r"<path to 85_5_10\training>")
val_85   = Path(r"<path to 85_5_10\validation>")
test_85  = Path(r"<path to 85_5_10\testing>")

train_90 = Path(r"<path to 90_5_5\training>")
val_90   = Path(r"<path to 90_5_5\validation>")
test_90  = Path(r"<path to 90_5_5\testing>")

# Name, training folder, validation folder, testing folder,
# validation fraction, testing fraction, split seed.
splits = [
    ("70_10_20", train_70, val_70, test_70, 0.10, 0.20, 42),
    ("85_5_10", train_85, val_85, test_85, 0.05, 0.10, 43),
    ("90_5_5", train_90, val_90, test_90, 0.05, 0.05, 44),
]

MOVE_IMAGES = True  # Set to False after the files have been split.

CLASS_NAMES = ["Normal", "Murmur"]  # Normal = 0, Murmur = 1
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 15
TRAIN_SEED = 123

### Move images to validation and testing

For each class, shuffle the original list once. Take the validation images from the start of that list and the testing images from the next slice. Both counts are calculated from the **original** number of training images. The remaining images stay in training. Rounding down makes the proportions approximate

In [ ]:
if MOVE_IMAGES:
    for name, train_dir, val_dir, test_dir, val_fraction, test_fraction, seed in splits:
        # Prevent a second run from moving more images into existing splits.
        if any(val_dir.rglob("*.png")) or any(test_dir.rglob("*.png")):
            raise ValueError(f"{name}: validation/testing already contain PNGs. "
                             "Set MOVE_IMAGES = False if splitting is complete.")

    for name, train_dir, val_dir, test_dir, val_fraction, test_fraction, seed in splits:
        print(f"\nSplit {name} (seed {seed})")
        rng = random.Random(seed)

        for class_name in CLASS_NAMES:
            images = sorted((train_dir / class_name).glob("*.png"))
            rng.shuffle(images)

            n_val = int(len(images) * val_fraction)
            n_test = int(len(images) * test_fraction)

            for image in images[:n_val]:
                shutil.move(image, val_dir / class_name / image.name)

            for image in images[n_val:n_val + n_test]:
                shutil.move(image, test_dir / class_name / image.name)

            print(f"{class_name}: {len(images) - n_val - n_test} training, "
                  f"{n_val} validation, {n_test} testing")


### Define three model configurations

The three models use two, three and four convolution/pooling blocks, respectively. They also use different numbers of filters, dropout rates and learning rates. These are three predefined combinations to try, not a random search. Reusing them in each split lets you compare corresponding configurations.

load_images() returns grayscale spectrograms resized to 64 × 64. The pixel values are normalized. They are divided by 255 so that the CNN receives values in [0, 1]. 
**Use the same preprocessing when testing the saved models later.** 


In [ ]:
configs = [
    {"name": "cnn_2_blocks", "filters": [4, 8],
     "dense_units": 16, "dropout": 0.20, "learning_rate": 0.001},
    {"name": "cnn_3_blocks", "filters": [8, 16, 24],
     "dense_units": 32, "dropout": 0.30, "learning_rate": 0.0005},
    {"name": "cnn_4_blocks", "filters": [8, 16, 24, 32],
     "dense_units": 32, "dropout": 0.40, "learning_rate": 0.0003},
]


def load_images(folder, shuffle):
    dataset = tf.keras.utils.image_dataset_from_directory(
        str(folder),
        labels="inferred",
        label_mode="binary",
        class_names=CLASS_NAMES,
        color_mode="grayscale",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=TRAIN_SEED,
    )
    return dataset.map(lambda images, labels: (images / 255.0, labels))


def build_model(config):
    layers = tf.keras.layers
    model = tf.keras.Sequential(name=config["name"])
    model.add(tf.keras.Input(shape=(*IMAGE_SIZE, 1)))

    for filters in config["filters"]:
        model.add(layers.Conv2D(filters, 3, activation="relu"))
        model.add(layers.MaxPooling2D(pool_size=2))

    model.add(layers.Flatten())
    model.add(layers.Dense(config["dense_units"], activation="relu"))
    model.add(layers.Dropout(config["dropout"]))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")],
    )
    return model

def plot_training_history(history, model_name):
    epochs = range(1, len(history.history['accuracy']) + 1)
    # Plot Accuracy
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history['accuracy'], 'b', label='Training Accuracy')
    plt.plot(epochs, history.history['val_accuracy'], 'r', label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    # Plot Recall
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history['recall'], 'b', label='Training Recall')
    plt.plot(epochs, history.history['val_recall'], 'r', label='Validation Recall')
    plt.title('Training and Validation Recall')
    plt.xlabel('Epochs')
    plt.ylabel('Recall')
    plt.legend()
    plt.show()
    plt.savefig(f"{model_name}_accuracy_loss_curves.png")


### Train and save all nine models

For each split, load training and validation once. Train the three CNNs and save each as a `.keras` file. At the same time, the following code will save `.txt` files containing the description of each model and a `.png` file containing the plot of the evolution of accuracy and loss throughout the training epochs. The files are save in the working directory of this jupyter notebook. Precision and recall refer to **Murmur (1)**, at a threshold of 0.5.

In [ ]:
print("Saving models in:", Path.cwd())

for name, train_dir, val_dir, test_dir, val_fraction, test_fraction, seed in splits:
    training_dataset = load_images(train_dir, shuffle=True)
    validation_dataset = load_images(val_dir, shuffle=False)

    for config in configs:
        filename = f"{name}_{config['name']}"
        print(f"\nTraining {filename}")
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(TRAIN_SEED)

        model = build_model(config)
        history = model.fit(
            training_dataset,
            validation_data=validation_dataset,
            epochs=EPOCHS,
        )
        model.save(filename + ".keras")
        plot_training_history(history, filename)

        with open(filename + ".txt", "w", encoding="utf-8") as report:
            report.write(f"Model file: {filename}.keras\n")
            report.write(f"Train/validation/test (%): {name.replace('_', '/')}\n")
            report.write(f"Split seed: {seed}; training seed: {TRAIN_SEED}\n")
            report.write(f"Classes: {CLASS_NAMES} (indices 0, 1)\n")
            report.write(f"Input: {IMAGE_SIZE}, grayscale, pixels divided by 255\n")
            report.write(f"Epochs: {EPOCHS}; batch size: {BATCH_SIZE}\n")
            report.write("Optimizer: Adam; loss: binary_crossentropy\n")
            report.write(f"CNN configuration: {config}\n")
            report.write("Blocks: Conv2D (3x3, valid, ReLU) + MaxPooling2D (2x2)\n")
            report.write("Head: Flatten, Dense (ReLU), Dropout, Dense (sigmoid)\n")
            report.write("\nArchitecture summary:\n")
            model.summary(print_fn=lambda line: report.write(line + "\n"))
            report.write("\nValidation at the last epoch (threshold 0.5):\n")
            for metric, values in history.history.items():
                if metric.startswith("val_"):
                    report.write(f"{metric}: {values[-1]:.4f}\n")


### Test the saved models

Save all the predictions in a < model name >_test_predictions.txt file. Save the confusion matrix and classification report in the preiously created < model name >.txt file. The confusion matrix and classification report are calculated at a threshold of 0.5.
Save the plots of the training and validation loss and accuracy in a png file.

In [ ]:
model_name = "<model name>" #e.g. : 70_10_20_cnn_2_blocks
test_dir = <variable containing the testing dataset of the split you want to use> # e.g. : test_70
threshold = 0.5

model = tf.keras.models.load_model(f"{model_name}.keras", compile=False)
testing_dataset = load_images(test_dir, shuffle=False)
# testing_dataset = load_images(test_dir, shuffle=False).map(prepare_resnet_input) # use only for the resnet50_murmur model, which requires the input to be preprocessed with the prepare_resnet_input function.

# Collect each score and its true label from the same batch.
y_true = []
scores = []

for images, labels in testing_dataset: #loops over the batches of images and labels in the testing dataset
    # Using def call(self, inputs, training=False) of the model instance returns the raw sigmoid scores. The training argument is set to False to avoid applying dropout during inference.
    batch_scores = model(images, training=False).numpy().ravel() 
    # .extend() is appropriate instead of .append() because .extend() adds each element of the iterable to the list, while .append() would add the entire iterable as a single element. Here, we want to add each score individually to the scores list.
    scores.extend(batch_scores)
    y_true.extend(labels.numpy().ravel().astype(int))

# Save the raw sigmoid scores before applying the threshold.
predictions_file = f"{model_name}_test_predictions.txt"
with open(predictions_file, "w", encoding="utf-8") as file:
    file.write("Image index\tTrue class\tMurmur score\n")
    for index, (label, score) in enumerate(zip(y_true, scores), start=1):
        file.write(f"{index}\t{CLASS_NAMES[label]}\t{score:.9g}\n")

# Normal = 0; Murmur = 1.
y_pred = (np.array(scores) >= threshold).astype(int)

report = classification_report(
    y_true, y_pred, labels=[0, 1], target_names=CLASS_NAMES, zero_division=0
)
matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])

results = (
    f"Test evaluation — threshold: {threshold}\n\n"
    f"Classification report:\n{report}\n"
    f"Confusion matrix (rows: true, columns: predicted; order: Normal, Murmur):\n"
    f"{matrix}\n"
)

# Append to the report created during training.
with open(f"{model_name}.txt", "a", encoding="utf-8") as file:
    file.write("\n" + results)

print(results)
print(f"Raw scores saved in: {predictions_file}")

### Hyperparameter Tuning

Train three CNNs on the same 70/10/20 split we created in the previous exercise. Each CNN will have three convolution–pooling blocks. The first two blocks will have two convolutional layers each. We use KerasTuner to vary the number of convolutions in the last block from one to three.
We will keep the input size, filter counts, learning rate, dropout, and number of epochs fixed so that depth is the architectural choice being tested.

Use GridSearch Tuner

In [ ]:
import keras_tuner as kt

In [ ]:
train_ds = load_images(train_70, shuffle=True)
val_ds = load_images(val_70, shuffle=False)


def build_tuner_model(hp):
    # This is the only hyperparameter in the search.
    last_block_convs = hp.Choice("last_block_convs", [1, 2, 3])

    layers = tf.keras.layers
    model = tf.keras.Sequential(name=f"70_10_20_KT_cnn_{4 + last_block_convs}_convs")
    model.add(tf.keras.Input(shape=(*IMAGE_SIZE, 1)))

    filters_per_block = [8, 16, 24]
    convs_per_block = [2, 2, last_block_convs]

    for filters, num_convs in zip(filters_per_block, convs_per_block):
        for _ in range(num_convs):
            model.add(layers.Conv2D(filters, 3, padding="valid", activation="relu"))
        model.add(layers.MaxPooling2D(pool_size=2))

    model.add(layers.Flatten())
    model.add(layers.Dense(32, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


tuner = kt.GridSearch(
    build_tuner_model,
    objective=kt.Objective("val_pr_auc", direction="max"),
    max_trials=3,
    directory="keras_tuner_runs",
    project_name="cnn_depth_70_10_20",
    overwrite=True,
)

tuner.search(train_ds, validation_data=val_ds, epochs=15)

# Load all three models, ranked by validation PR-AUC.
for rank, model in enumerate(tuner.get_best_models(num_models=3), start=1):
    metrics = model.evaluate(val_ds, return_dict=True, verbose=0)
    model.save(f"{model.name}.keras")
    
    with open(model.name + ".txt", "w", encoding="utf-8") as report:
        report.write(f"Model file: {model.name}.keras\n")
        report.write(f"Train/validation/test (%): 70/10/20\n")
        report.write(f"Classes: {CLASS_NAMES} (indices 0, 1)\n")
        report.write(f"Input: {IMAGE_SIZE}, grayscale, pixels divided by 255\n")
        report.write(f"Epochs: 15; batch size: {BATCH_SIZE}\n")
        report.write("Optimizer: Adam; loss: binary_crossentropy\n")
        report.write("\nArchitecture summary:\n")
        model.summary(print_fn=lambda line: report.write(line + "\n"))
        report.write("\nMetrics :\n")
        report.write(
            f"{rank}. {model.name} | parameters: {model.count_params():,} | "
            f"val PR-AUC: {metrics['pr_auc']:.3f} | "
            f"val precision: {metrics['precision']:.3f} | "
            f"val recall: {metrics['recall']:.3f}"
        )


### Transfer Learning

Build a CNN which uses the first residual block of a ResNet50 model pretrained on ImageNet and a custom classification head for detecting murmurs

In [ ]:
def prepare_resnet_input(images, labels):
    # load_images() returns grayscale pixels in [0, 1].
    # tf.image.grayscale_to_rgb does not paint back the image. I just copies the grayscale channel in all 3 RGB channels.
    rgb = tf.image.grayscale_to_rgb(images * 255.0)
    images = tf.keras.applications.resnet.preprocess_input(rgb)
    return images, labels

train_ds = load_images(train_70, shuffle=True).map(prepare_resnet_input)
val_ds = load_images(val_70, shuffle=False).map(prepare_resnet_input)

def build_resnet_classifier(hp):
    resnet = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMAGE_SIZE, 3),
    )
    resnet.trainable = False

    # Connect the head directly to the retained graph, without a nested model.
    x = resnet.get_layer("conv2_block1_out").output

    projection_filters = hp.Choice("projection_filters", [8, 16])
    dense_units = hp.Choice("dense_units", [16, 32])

    layers = tf.keras.layers
    x = layers.Conv2D(projection_filters, 1, activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=4)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    score = layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(resnet.input, score, name="resnet50_murmur_detection")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


tuner = kt.GridSearch(
    build_resnet_classifier,
    objective=kt.Objective("val_pr_auc", direction="max"),
    max_trials=4,
    directory="keras_tuner_runs",
    project_name="resnet50_murmur_head",
    overwrite=True,
)

tuner.search(train_ds, validation_data=val_ds, epochs=15)
tuner.results_summary(num_trials=4)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]

print("Best head:", best_hp.values)
best_model.summary()
best_model.save("resnet50_murmur.keras")
